In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats import weightstats as stests
import os

os.makedirs('../../output/figs', exist_ok=True)
os.makedirs('../../output/tables', exist_ok=True)

# 1. Load Data
print("Loading data...")
df = pd.read_csv('../../output/04_Regression_handled.csv')
df['AR'] = df['R_i'] - (df['alpha'] + df['beta'] * df['R_m'])

# ==========================================
# FIG 3: Daily AAR Time Series with 95% CI
# ==========================================
print("Generating AAR Time Series...")
event_data = df[(df['day'] >= -5) & (df['day'] <= 30)]
aar_daily = event_data.groupby('day')['AR'].agg(['mean', 'std', 'count']).reset_index()
aar_daily['se'] = aar_daily['std'] / np.sqrt(aar_daily['count'])
aar_daily['ci_upper'] = aar_daily['mean'] + (1.96 * aar_daily['se'])
aar_daily['ci_lower'] = aar_daily['mean'] - (1.96 * aar_daily['se'])

plt.figure(figsize=(10, 5))
plt.plot(aar_daily['day'], aar_daily['mean'] * 100, color='crimson', label='AAR', linewidth=2)
plt.fill_between(aar_daily['day'], aar_daily['ci_lower'] * 100, aar_daily['ci_upper'] * 100, color='crimson', alpha=0.2, label='95% CI')
plt.axvline(0, color='black', linestyle='--')
plt.axhline(0, color='gray', linewidth=1)
plt.title('Daily Average Abnormal Returns (AAR) with 95% Confidence Intervals')
plt.xlabel('Days Relative to Landfall')
plt.ylabel('AAR (%)')
plt.legend()
plt.tight_layout()
plt.savefig('../../output/figs/fig3_aar_timeseries.png', dpi=300)
plt.close()

# ==========================================
# TABLE 6: BMP Test & Multiple Testing Corrections
# ==========================================
print("Running BMP and Multiple Testing Corrections...")
full_window = df[(df['day'] >= -5) & (df['day'] <= 30)]
stock_cars = full_window.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
stock_cars.rename(columns={'AR': 'CAR'}, inplace=True)

results = []
for sector in stock_cars['sector'].unique():
    sector_cars = stock_cars[stock_cars['sector'] == sector]['CAR']
    n = len(sector_cars)
    if n > 2:
        mean_car = sector_cars.mean()
        t_stat, p_val = stests.ztest(sector_cars, value=0)
        results.append({'Sector': sector, 'N': n, 'Mean_CAR': mean_car, 'Raw_P': p_val, 'T_Stat': t_stat})

results_df = pd.DataFrame(results)
reject_bonf, p_bonf, _, _ = multipletests(results_df['Raw_P'], alpha=0.05, method='bonferroni')
reject_fdr, p_fdr, _, _ = multipletests(results_df['Raw_P'], alpha=0.05, method='fdr_bh')

results_df['Bonferroni_P'] = p_bonf
results_df['FDR_Q_Value'] = p_fdr
results_df['Significant_FDR'] = reject_fdr
results_df.to_csv('../../output/tables/table6_multiple_testing.csv', index=False)

# ==========================================
# FIG 4: Sector CAR Heatmap
# ==========================================
print("Generating Sector Heatmap...")
windows = {
    'Pre-Event [-5, -1]': df[(df['day'] >= -5) & (df['day'] <= -1)],
    'Landfall [0, 0]': df[df['day'] == 0],
    'Aftermath [0, +5]': df[(df['day'] >= 0) & (df['day'] <= 5)],
    'Full [-5, +30]': df[(df['day'] >= -5) & (df['day'] <= 30)]
}

heatmap_data = pd.DataFrame(index=stock_cars['sector'].unique())

for name, win_df in windows.items():
    win_cars = win_df.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
    sector_means = win_cars.groupby('sector')['AR'].mean() * 100 
    heatmap_data[name] = sector_means

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="RdYlGn", center=0, cbar_kws={'label': 'Mean CAR (%)'})
plt.title('Sector Cumulative Abnormal Returns by Event Window')
plt.tight_layout()
plt.savefig('../../output/figs/fig4_sector_heatmap.png', dpi=300)
plt.close()

# ==========================================
# TABLE 7: Cross-Sectional Regression
# ==========================================
print("Running Cross-Sectional Regression...")

# 1. Merge and Feature Engineering
reg_data = stock_cars.copy()
firm_chars = df.groupby('symbol').agg({'beta': 'first', 'volume': 'mean'}).reset_index()
reg_data = pd.merge(reg_data, firm_chars, on='symbol')
reg_data['log_volume'] = np.log1p(reg_data['volume'])

# 2. Categorical Encoding
reg_data = pd.get_dummies(reg_data, columns=['sector'], drop_first=True)

# 3. Define y and X
y = reg_data['CAR'] * 100 
dummy_cols = [col for col in reg_data.columns if col.startswith('sector_')]
X = reg_data[['beta', 'log_volume'] + dummy_cols].astype(float)
X = sm.add_constant(X)

# 4. CRITICAL: Drop rows with NaNs/Infs before fitting
# This prevents the MissingDataError you saw earlier
valid_idx = X.replace([np.inf, -np.inf], np.nan).dropna().index
X_clean = X.loc[valid_idx]
y_clean = y.loc[valid_idx]

# 5. Fit
model = sm.OLS(y_clean, X_clean).fit()

with open('../../output/tables/table7_regression_summary.txt', 'w') as f:
    f.write(model.summary().as_text())

print("SUCCESS: All advanced conference upgrades generated!")

Loading data...
Generating AAR Time Series...
Running BMP and Multiple Testing Corrections...
Generating Sector Heatmap...
Running Cross-Sectional Regression...
SUCCESS: All advanced conference upgrades generated!
